In [ ]:
# ================================================================ half-wave plates
# Run this cell first, then the simulator cell below. Re-run both after changing anything here.
# Angles are fast-axis angles in degrees from H (the table plane).
#   HWP0 input, before PBS1          HWP3 arm C, between PBS3 and M2
#   HWP1 between PBS1 and PBS3       HWP4 between PBS2 and PBS4
#   HWP2 arm A, between M1 and PBS2  HWP5 between PBS4 and the analyzer PBS
import importlib
import interferometer_sim
importlib.reload(interferometer_sim)
from interferometer_sim import equal_intensity_angles, jones_paths, HWP_NAMES, HWP_WHERE
import numpy as np

LASER_POL_DEG = 0.0   # laser polarization angle from H
PBS_LEAK = 0.0        # PBS leakage, e.g. 1e-3 for a 1000:1 cube

HWP_DEG = equal_intensity_angles(LASER_POL_DEG)   # equal intensity from A, B and C
# Override individual plates here, e.g.
# HWP_DEG["HWP5"] = 0.0

for n in HWP_NAMES:
    print(f"{n}  {HWP_DEG[n]:7.2f}°   {HWP_WHERE[n]}")
J = jones_paths(HWP_DEG, LASER_POL_DEG, PBS_LEAK)
P = {p: float(np.sum(np.abs(v)**2)) for p, v in J.items()}
print("\npower to camera (fraction of laser):", "  ".join(f"{p}: {100*P[p]:.2f}%" for p in P),
      f"  total {100*sum(P.values()):.1f}%")


# Three-path interferometer – live fringe simulator

Interactive model of the camera pattern from the three-path PBS interferometer (λ₀ = 810 nm).
The three beams reach the camera **side by side** along x (A left, B middle, C right) and slightly tilted toward each other, so where they overlap they form straight fringes.

| path | launching optic (stage + tilt) | phase shifter |
|---|---|---|
| A | PBS1 (reflected arm) | slide θ_A |
| B | PBS3 (reflected arm) | slide θ_B |
| C | M2 (right-hand mirror) | – (reference arm) |

The simulated camera is the PI-MAX 4 sensor: **1024 × 1024 pixels** (12.8 µm pixels ≈ 13.1 mm across). `VIEW_PX` sets whether the full sensor or a centred 512 / 256 px window is shown; fringe periods are reported in pixels.

**Polarization** is modelled with Jones matrices: each HWP rotates the polarization by twice its angle, each PBS transmits H and reflects V (optional leakage). The light each path brings to the camera therefore depends on the six HWP angles. The angles are set in the **half-wave plate cell at the top**. By default it uses the equal-intensity angles HWP0 = 17.63°, HWP1 = 22.5°, HWP2 = HWP3 = 45°, HWP4 = 22.5°, HWP5 = 17.63° (for the laser polarization `LASER_POL_DEG`, default H), giving 1/9 of the laser power from each path, the most possible with equal intensities. Brightness is normalised to that ideal, so a mis-set plate shows up as a dimmer or lopsided pattern.

**Setup parameters** (wavelength, slide thickness, beam waist, pixel size, view size, lever arms) are constants at the top of the simulator cell; laser polarization, PBS leakage and HWP angles are in the half-wave plate cell. Edit them and re-run the cells (half-wave plate cell first).

**What each control does**
- **Slide θ** – tilting a microscope slide changes that path's phase (and shifts the beam slightly sideways).
- **Stage** – translating PBS1 / PBS3 / M2 along the incoming beam moves that beam sideways on the camera *and* lengthens the path by the same amount. At 810 nm, 1 µm of stage is ≈ 7.8 rad, so stages are for overlap; set phases with the slides. Set `STAGE_CHANGES_PHASE = False` to see only the position effect.
- **Tilt x / y** – tilting the optic by δ steers the beam by 2δ: this sets the fringe spacing and orientation, and moves the beam by 2δ·L on the camera (L = lever arm to the camera, set in `Setup.lever_mm`).

The physics lives in `interferometer_sim.py` (`Setup`), so the same model can be used in analysis scripts. Assumptions: Gaussian beams, ideal mirrors, BK7-like slide index.


In [ ]:
import io
import numpy as np
import ipywidgets as W
from IPython.display import display
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

import importlib
import interferometer_sim
importlib.reload(interferometer_sim)  # pick up edits to the module without restarting the kernel
from interferometer_sim import Setup, PATHS, LAUNCH_OPTIC, slide_slope, slide_walk, draw_setup

assert "HWP_DEG" in globals(), "Run the half-wave plate cell at the top of the notebook first."

# ================================================================ setup (edit here)
LAMBDA_NM = 810.0            # laser wavelength
SLIDE_T_MM = 1.0             # microscope-slide thickness
WAIST_MM = 1.0               # beam waist at the camera
STAGE_CHANGES_PHASE = True   # translating PBS1 / PBS3 / M2 also lengthens that path
PIXEL_UM = 12.8              # PI-MAX 4 pixel pitch (1024 x 1024 sensor)
VIEW_PX = 512                # camera window shown: 1024 (full sensor), 512 or 256
LEVER_MM = {"A": 600.0, "B": 450.0, "C": 300.0}   # launch optic -> camera distance


def make_setup():
    # HWP_DEG, LASER_POL_DEG and PBS_LEAK come from the half-wave plate cell at the top
    return Setup(lam_nm=LAMBDA_NM, input_pol_deg=LASER_POL_DEG, slide_t_mm=SLIDE_T_MM,
                 waist_mm=WAIST_MM, pbs_leak=PBS_LEAK, stage_changes_phase=STAGE_CHANGES_PHASE,
                 pixel_um=PIXEL_UM, lever_mm=dict(LEVER_MM), hwp_deg=dict(HWP_DEG))
# ================================================================================


setup = make_setup()
COLORS = {"A": "#4fc3f7", "B": "#ffd54f", "C": "#ef5350"}

# ---------------------------------------------------------------- widgets
def fslider(desc, v, lo, hi, step, fmt):
    return W.FloatSlider(value=v, min=lo, max=hi, step=step, description=desc,
                         readout_format=fmt, continuous_update=True,
                         style={"description_width": "110px"}, layout=W.Layout(width="420px"))

thA = fslider("slide θ_A (°)", setup.theta_A_deg, 0, 30, 0.001, ".3f")
thB = fslider("slide θ_B (°)", setup.theta_B_deg, 0, 30, 0.001, ".3f")

stage, tx, ty, blk = {}, {}, {}, {}
tabs = []
for p in PATHS:
    L = setup.launch[p]
    stage[p] = fslider(f"{LAUNCH_OPTIC[p]} stage (mm)", L.stage_mm, -1.5, 1.5, 0.0005, ".4f")
    tx[p] = fslider(f"{LAUNCH_OPTIC[p]} tilt x (mrad)", L.tilt_x_mrad, -2, 2, 0.005, ".3f")
    ty[p] = fslider(f"{LAUNCH_OPTIC[p]} tilt y (mrad)", L.tilt_y_mrad, -2, 2, 0.005, ".3f")
    blk[p] = W.Checkbox(value=L.blocked, description=f"block path {p}")
    tabs.append(W.VBox([stage[p], tx[p], ty[p], blk[p]]))
beam_tabs = W.Tab(children=tabs)
for i, p in enumerate(PATHS):
    beam_tabs.set_title(i, f"{p}  ({LAUNCH_OPTIC[p]})")

info = W.HTML()
img = W.Image(format="png")

# ---------------------------------------------------------------- figure
fig = Figure(figsize=(11, 9.6), dpi=90)
FigureCanvasAgg(fig)
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.35], height_ratios=[1.15, 1], wspace=0.25, hspace=0.12,
                      left=0.06, right=0.98, top=0.98, bottom=0.06)
ax_set = fig.add_subplot(gs[0, :])
ax_cam = fig.add_subplot(gs[1, 0])
ax_pro = fig.add_subplot(gs[1, 1])
grid = {}


def make_grid():
    """Pixel window currently shown, and the mm coordinates of every pixel in it."""
    px = setup.pixel_window(VIEW_PX)
    c = setup.px_to_mm(px)
    X, Y = np.meshgrid(c, c)
    grid.update(px=px, c=c, X=X, Y=Y, row=len(px) // 2,
                extent=[px[0] - 0.5, px[-1] + 0.5, px[0] - 0.5, px[-1] + 0.5])


make_grid()
im = ax_cam.imshow(np.zeros((len(grid["px"]),) * 2), extent=grid["extent"], origin="lower",
                   cmap="inferno", vmin=0, vmax=1, interpolation="nearest")
ax_cam.set_title("PI-MAX 4 (simulated)")
ax_cam.set_xlabel("x (pixel)"); ax_cam.set_ylabel("y (pixel)")
hline = ax_cam.axhline(grid["px"][grid["row"]], color="w", lw=0.6, ls=":", alpha=0.6)
line_now, = ax_pro.plot(grid["px"], 0 * grid["c"], color="#5b45e0", lw=2, label="now")
env_lines = {p: ax_pro.plot(grid["px"], 0 * grid["c"], color=COLORS[p], lw=1, alpha=0.7, label=f"|E_{p}|²")[0] for p in PATHS}
ax_pro.set_ylim(0, 1.05)
ax_pro.set_xlabel("x (pixel)"); ax_pro.set_ylabel("I / I_max")
ax_pro.set_title("cross-section through the centre row")
ax_pro.grid(alpha=0.3); ax_pro.legend(loc="upper right", fontsize=8)


def read_widgets():
    setup.theta_A_deg, setup.theta_B_deg = thA.value, thB.value
    for p in PATHS:
        L = setup.launch[p]
        L.stage_mm, L.tilt_x_mrad, L.tilt_y_mrad, L.blocked = stage[p].value, tx[p].value, ty[p].value, blk[p].value


def redraw(*_):
    read_widgets()
    X, Y, px, c, row = grid["X"], grid["Y"], grid["px"], grid["c"], grid["row"]
    draw_setup(ax_set, setup)
    Imax = setup.max_intensity(X, Y)
    I = setup.intensity(X, Y) / Imax
    im.set_data(I); im.set_extent(grid["extent"])
    ax_cam.set_xlim(grid["extent"][:2]); ax_cam.set_ylim(grid["extent"][2:])
    hline.set_ydata([px[row], px[row]])
    prof = I[row]
    line_now.set_data(px, prof)
    ax_pro.set_xlim(px[0], px[-1])
    ax_pro.set_ylim(0, max(1.05, 1.05 * float(prof.max())))
    for p in PATHS:
        b = setup.beam(p)
        on = not b["blocked"]
        env = b["amp"]**2 * np.exp(-2 * ((c - b["x0"])**2 + (c[row] - b["y0"])**2) / setup.waist_mm**2) / Imax
        env_lines[p].set_data(px, env); env_lines[p].set_visible(on)
    buf = io.BytesIO(); fig.savefig(buf, format="png"); img.value = buf.getvalue()

    ph = setup.phases()
    fp, fpx = setup.fringe_periods(), setup.fringe_periods_px()
    fp_txt = ", ".join(f"{k}: {fpx[k]:.1f} px ({v:.3f} mm)" for k, v in fp.items()) or "–"
    sensor_mm = setup.npix * setup.pixel_um * 1e-3
    ang = ", ".join(f"{p}: {setup.beam(p)['angle_x_mrad']:+.2f}" for p in PATHS)
    pw = setup.camera_powers()
    pw_txt = " &nbsp; ".join(f"{p}: {100*pw[p]:.2f}%" for p in PATHS)
    spread = (max(pw.values()) - min(pw.values())) / max(max(pw.values()), 1e-12)
    pw_flag = "equal" if spread < 0.01 else f"<span style='color:#c0392b'>unequal (spread {100*spread:.0f}%)</span>"
    info.value = (
        "<div style='font-family:monospace;font-size:13px;line-height:1.6'>"
        f"power to camera (of laser): {pw_txt} &nbsp; total {100*sum(pw.values()):.1f}% &nbsp; {pw_flag}<br>"
        f"φ_A = {ph['A']:.3f} rad &nbsp; φ_B = {ph['B']:.3f} rad &nbsp; φ_C = {ph['C']:.3f} rad (mod 2π)<br>"
        f"slide dφ/dθ: A {slide_slope(setup.theta_A_deg, setup.lam_nm, setup.slide_t_mm):.1f}, "
        f"B {slide_slope(setup.theta_B_deg, setup.lam_nm, setup.slide_t_mm):.1f} rad/° &nbsp; "
        f"slide beam walk: A {1e3*slide_walk(setup.theta_A_deg, setup.lam_nm, setup.slide_t_mm):.0f} µm, "
        f"B {1e3*slide_walk(setup.theta_B_deg, setup.lam_nm, setup.slide_t_mm):.0f} µm<br>"
        f"beam angles x (mrad): {ang}<br>"
        f"fringe periods: {fp_txt}<br>"
        f"sensor: {setup.npix} × {setup.npix} px, {setup.pixel_um:.1f} µm pixels = {sensor_mm:.2f} mm across; "
        f"beam waist = {setup.waist_mm / (setup.pixel_um * 1e-3):.0f} px<br>"
        f"stage phase: 1 µm of stage = {2*np.pi*1e3/setup.lam_nm:.2f} rad</div>"
    )


for w in [thA, thB] + [d[p] for d in (stage, tx, ty, blk) for p in PATHS]:
    w.observe(redraw, names="value")

controls = W.VBox([
    W.HTML("<b>Phase shifters</b>"), thA, thB,
    W.HTML("<b>Launch optics</b>"), beam_tabs,
])
redraw()
display(W.VBox([img, controls, info]))


In [ ]:
# ================================================================ phase compensation
# A 630 nm reference laser shares all three paths with the 810 nm beam (dichroic DM1 at the input,
# achromatic HWPs). The beam blocks are short-pass filters, so the reference always sees A, B and C.
# After the analyzer, dichroic DM2 sends the 630 nm fringes onto two pinholes, each with a photodiode.
# The PC holds both photodiode readings at their values at lock time by turning the two slides
# through the Newport XPS (integral loop using the 2 x 2 matrix dI/dtheta, measured at start-up).
# Two photodiodes are needed: one reading cannot pin down two phases (shown below with N_PD = 1).
# The XPS is modelled with a command latency, a maximum stage speed and a minimum step.
# Uses the current slider settings from the simulator cell.
import copy
import matplotlib.pyplot as plt
from interferometer_sim import draw_setup, simulate_intensity_lock, pinhole_intensity, PATHS

assert "setup" in globals(), "Run the simulator cell first."

# ---------------------------------------------------------------- lock settings (edit here)
REF_NM = 630.0                                # reference laser wavelength
PINHOLE_UM = 50.0                             # pinhole diameter
LOOP_HZ = 10.0                                # photodiode read + XPS command rate
XPS_LATENCY_S = 0.05                          # PC -> XPS -> stage starts moving
XPS_VEL_DEG_S = 2.0                           # rotation-stage speed
XPS_STEP_DEG = 0.001                          # rotation-stage minimum step
GAIN = 0.5                                    # integral gain (fraction of the error corrected per update)
PD_NOISE = 0.002                              # photodiode noise (fraction of the reading)
DURATION_S = 300.0                            # simulated time
WALK_NM = 3.0                                 # random path-length drift per arm, nm per sqrt(s)
RATE_NM_S = {"A": 2.0, "B": -1.0, "C": 0.5}   # steady thermal drift per arm (nm/s)

kw = dict(ref_nm=REF_NM, pinhole_um=PINHOLE_UM, duration_s=DURATION_S, loop_hz=LOOP_HZ,
          latency_s=XPS_LATENCY_S, vel_deg_s=XPS_VEL_DEG_S, step_deg=XPS_STEP_DEG, gain=GAIN,
          walk_nm=WALK_NM, rate_nm_s=RATE_NM_S, pd_noise=PD_NOISE)
res2 = simulate_intensity_lock(setup, n_pd=2, **kw)
res1 = simulate_intensity_lock(setup, n_pd=1, **kw)   # same drift, one photodiode only
t = res2["t"]
rms = lambda e: np.sqrt(np.mean(e**2, axis=0))

print("pinholes (x from the sensor centre, y = 0):",
      ", ".join(f"PD{i+1} {1e3*x:+.0f} µm" for i, x in enumerate(res2["x_pinholes"])))
print(f"\n810 nm phase error over {t[-1]:.0f} s       phi_A-phi_C   phi_B-phi_C")
for lab, e in [("no compensation (RMS)", res2["err_free"]), ("1 photodiode (RMS)", res1["err_lock"]),
               ("2 photodiodes (RMS)", res2["err_lock"]), ("2 photodiodes (max)", None)]:
    v = np.abs(res2["err_lock"]).max(0) if e is None else rms(e)
    print(f"  {lab:<24}{v[0]:9.4f} rad  {v[1]:9.4f} rad")
print(f"slide travel: A {np.ptp(res2['theta_A']):.3f}°, B {np.ptp(res2['theta_B']):.3f}°")

# ---------------------------------------------------------------- schematic
fig1, ax = plt.subplots(figsize=(12, 6.4))
draw_setup(ax, res2["locked"], lock=True)
ax.set_title("Setup with active phase compensation", fontsize=12)
fig1.tight_layout()
plt.show()

# ---------------------------------------------------------------- lock performance
ref = copy.deepcopy(res2["ref"])
for L in ref.launch.values():
    L.drift_rad = 0.0
ref.theta_A_deg, ref.theta_B_deg = setup.theta_A_deg, setup.theta_B_deg
fig2, axs = plt.subplots(3, 2, figsize=(12, 12))

xl = np.linspace(-2.5, 2.5, 1001)
Il = pinhole_intensity(ref, xl, 0.0)
axs[0, 0].plot(1e3 * xl, Il / Il.max(), color="#ff8c00", lw=1.5)
for i, x in enumerate(res2["x_pinholes"]):
    axs[0, 0].axvline(1e3 * x, color="#0d8774", ls="--", lw=1.2)
    axs[0, 0].text(1e3 * x, 1.03, f"PD{i+1}", ha="center", color="#0d8774", fontsize=9, fontweight="bold")
axs[0, 0].set_ylim(0, 1.12); axs[0, 0].grid(alpha=0.3)
axs[0, 0].set_xlabel("x at the pinhole plane (µm)"); axs[0, 0].set_ylabel("I (norm.)")
axs[0, 0].set_title(f"{REF_NM:.0f} nm fringes and pinhole positions")

for i, c in enumerate(("#0d8774", "#7b4fd6")):
    axs[0, 1].plot(t, res2["pd"][:, i], color=c, lw=1, label=f"PD{i+1}")
axs[0, 1].axhline(1, color="k", lw=0.8)
axs[0, 1].set_xlabel("time (s)"); axs[0, 1].set_ylabel("PD / setpoint")
axs[0, 1].grid(alpha=0.3); axs[0, 1].legend(fontsize=8)
axs[0, 1].set_title("photodiode signals (held at their setpoints)")

pairs = [("φ_A − φ_C", "#4fc3f7"), ("φ_B − φ_C", "#ef5350")]
ef = np.unwrap(res2["err_free"], axis=0)
e1 = np.unwrap(res1["err_lock"], axis=0)
for i, (lab, c) in enumerate(pairs):
    axs[1, 0].plot(t, ef[:, i], color=c, lw=1, alpha=0.45, label=f"{lab}, no compensation")
    axs[1, 0].plot(t, e1[:, i], color=c, lw=1.3, ls=":", label=f"{lab}, 1 photodiode")
    axs[1, 0].plot(t, res2["err_lock"][:, i], color=c, lw=2.2, label=f"{lab}, 2 photodiodes")
axs[1, 0].set_xlabel("time (s)"); axs[1, 0].set_ylabel("810 nm phase error (rad)")
axs[1, 0].grid(alpha=0.3); axs[1, 0].legend(fontsize=7, ncol=2)
axs[1, 0].set_title("phase error at the PI-MAX")

for i, (lab, c) in enumerate(pairs):
    axs[1, 1].plot(t, 1e3 * res2["err_lock"][:, i], color=c, lw=1, label=lab)
axs[1, 1].set_xlabel("time (s)"); axs[1, 1].set_ylabel("residual error (mrad)")
axs[1, 1].grid(alpha=0.3); axs[1, 1].legend(fontsize=8)
axs[1, 1].set_title("2-photodiode lock: residual (zoom)")

axs[2, 0].plot(t, res2["theta_A"], color="#4fc3f7", lw=2, label="slide A")
axs[2, 0].plot(t, res2["theta_B"], color="#ffd54f", lw=2, label="slide B")
axs[2, 0].set_xlabel("time (s)"); axs[2, 0].set_ylabel("slide angle (°)")
axs[2, 0].grid(alpha=0.3); axs[2, 0].legend(fontsize=8)
axs[2, 0].set_title("slide angles set by the XPS")

lat = np.array([0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0])
for g, c in ((0.2, "#5b45e0"), (GAIN, "#0d8774")):
    r_ = [np.max(rms(simulate_intensity_lock(setup, n_pd=2, **{**kw, "duration_s": 60.0, "latency_s": L,
                                                               "gain": g})["err_lock"])) for L in lat]
    axs[2, 1].semilogy(lat, r_, "o-", color=c, label=f"gain {g}")
axs[2, 1].axvline(XPS_LATENCY_S, color="k", ls=":", lw=1)
axs[2, 1].set_xlabel("XPS latency (s)"); axs[2, 1].set_ylabel("810 nm residual RMS (rad)")
axs[2, 1].grid(alpha=0.3, which="both"); axs[2, 1].legend(fontsize=8)
axs[2, 1].set_title(f"effect of XPS lag (loop at {LOOP_HZ:.0f} Hz)")
fig2.tight_layout()
plt.show()

# ---------------------------------------------------------------- PI-MAX cross-section at the end
start = copy.deepcopy(res2["free"])
for L in start.launch.values():
    L.drift_rad = 0.0
start.theta_A_deg, start.theta_B_deg = setup.theta_A_deg, setup.theta_B_deg
fig3, ax3 = plt.subplots(figsize=(12, 3.6))
pxm = start.pixel_window(VIEW_PX)
cm = start.px_to_mm(pxm)
Xm, Ym = np.meshgrid(cm, cm[len(cm) // 2: len(cm) // 2 + 1])
norm = start.max_intensity(*np.meshgrid(cm, cm))
for s_, lab, sty in [(start, "at lock (target)", dict(color="k", lw=2.5, alpha=0.35)),
                     (res2["free"], f"after {t[-1]:.0f} s, no compensation", dict(color="#c0392b", lw=1.2)),
                     (res1["locked"], f"after {t[-1]:.0f} s, 1 photodiode", dict(color="#e08e0b", lw=1.2, ls=":")),
                     (res2["locked"], f"after {t[-1]:.0f} s, 2 photodiodes", dict(color="#0d8774", lw=1.5, ls="--"))]:
    ax3.plot(pxm, s_.intensity(Xm, Ym)[0] / norm, label=lab, **sty)
ax3.set_xlim(pxm[0], pxm[-1]); ax3.set_xlabel("x (pixel)"); ax3.set_ylabel("I / I_max")
ax3.set_title("PI-MAX cross-section (810 nm)"); ax3.grid(alpha=0.3); ax3.legend(fontsize=8)
fig3.tight_layout()
plt.show()


In [ ]:
# ================================================================ live view (slow motion)
# Runs the 2-photodiode lock step by step and redraws after every frame, so you can follow one
# loop cycle at a time: a path-length kick moves the 630 nm fringes, the photodiodes leave their
# setpoints, the PC sends a correction, the XPS waits out its latency, the slides turn, and the
# error returns to zero. Uses the lock settings from the phase-compensation cell above.
# Stop early with the interrupt (square) button.
import io, time
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
from interferometer_sim import IntensityLockSim, pinhole_intensity

assert "setup" in globals(), "Run the simulator cell first."
_g = globals().get

# ---------------------------------------------------------------- live settings (edit here)
LIVE_DURATION_S = 8.0                     # simulated time to show
SIM_S_PER_FRAME = 0.01                    # simulated time per frame (smaller = slower)
FRAME_DELAY_S = 0.0                       # extra pause per frame (larger = slower)
WINDOW_S = 4.0                            # width of the scrolling time plots
KICKS_NM = [(1.0, "A", 80.0), (3.5, "B", -60.0), (6.0, "C", 50.0)]   # (time s, arm, path jump nm)

sim = IntensityLockSim(setup, n_pd=2, ref_nm=_g("REF_NM", 630.0), pinhole_um=_g("PINHOLE_UM", 50.0),
                       loop_hz=_g("LOOP_HZ", 10.0), latency_s=_g("XPS_LATENCY_S", 0.05),
                       vel_deg_s=_g("XPS_VEL_DEG_S", 2.0), step_deg=_g("XPS_STEP_DEG", 0.001),
                       gain=_g("GAIN", 0.5), pd_noise=_g("PD_NOISE", 0.002), walk_nm=_g("WALK_NM", 3.0),
                       rate_nm_s=_g("RATE_NM_S", None), kicks_nm=KICKS_NM)
steps_per_frame = max(1, int(round(SIM_S_PER_FRAME / sim.dt)))

# ---------------------------------------------------------------- figure (built once, updated per frame)
fig = Figure(figsize=(12, 9.2), dpi=80)
FigureCanvasAgg(fig)
gs = fig.add_gridspec(3, 2, hspace=0.45, wspace=0.22, left=0.07, right=0.98, top=0.95, bottom=0.06)
a_fr, a_pd = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
a_er, a_th = fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])
a_cam, a_txt = fig.add_subplot(gs[2, 0]), fig.add_subplot(gs[2, 1])

xl = np.linspace(-2.5, 2.5, 401)
I0 = pinhole_intensity(sim.ref, xl, 0.0); Inorm = I0.max()
a_fr.plot(1e3 * xl, I0 / Inorm, color="0.6", ls="--", lw=1, label="at lock")
l_fr, = a_fr.plot(1e3 * xl, I0 / Inorm, color="#ff8c00", lw=1.8, label="now")
pd_dots = []
for i, x in enumerate(sim.x_ph):
    a_fr.axvline(1e3 * x, color="#0d8774", ls=":", lw=1)
    a_fr.text(1e3 * x, 1.12, f"PD{i+1}", ha="center", color="#0d8774", fontsize=9, fontweight="bold")
    pd_dots.append(a_fr.plot([1e3 * x], [0], "o", color="#0d8774", ms=7)[0])
a_fr.set_ylim(0, 1.25); a_fr.set_xlabel("x at the pinhole plane (µm)"); a_fr.set_ylabel("I (norm.)")
a_fr.set_title("630 nm fringes on the pinholes"); a_fr.legend(fontsize=8, loc="upper right"); a_fr.grid(alpha=0.3)

l_pd = [a_pd.plot([], [], color=c, lw=1.5, label=f"PD{i+1}")[0] for i, c in enumerate(("#0d8774", "#7b4fd6"))]
a_pd.axhline(1, color="k", lw=0.8); a_pd.set_ylabel("PD / setpoint"); a_pd.set_xlabel("time (s)")
a_pd.set_title("photodiodes (the lock pushes these back to 1)"); a_pd.legend(fontsize=8); a_pd.grid(alpha=0.3)

pairs = (("φ_A − φ_C", "#4fc3f7"), ("φ_B − φ_C", "#ef5350"))
l_ef = [a_er.plot([], [], color=c, lw=1, alpha=0.45, label=f"{lab}, no lock")[0] for lab, c in pairs]
l_el = [a_er.plot([], [], color=c, lw=2, label=f"{lab}, locked")[0] for lab, c in pairs]
a_er.axhline(0, color="k", lw=0.8); a_er.set_ylabel("810 nm phase error (rad)"); a_er.set_xlabel("time (s)")
a_er.set_title("phase error at the PI-MAX"); a_er.legend(fontsize=7, ncol=2, loc="upper left"); a_er.grid(alpha=0.3)

l_cmd = [a_th.plot([], [], color=c, lw=1.2, ls="--", label=f"slide {p} commanded")[0]
         for p, c in (("A", "#4fc3f7"), ("B", "#e0a800"))]
l_pos = [a_th.plot([], [], color=c, lw=2, label=f"slide {p} actual")[0] for p, c in (("A", "#4fc3f7"), ("B", "#e0a800"))]
a_th.axhline(0, color="k", lw=0.8); a_th.set_ylabel("slide angle change (mdeg)"); a_th.set_xlabel("time (s)")
a_th.set_title("XPS: command vs actual (latency, then motion)"); a_th.legend(fontsize=7, ncol=2); a_th.grid(alpha=0.3)

pxm = sim.locked.pixel_window(VIEW_PX); cm = sim.locked.px_to_mm(pxm)
Xm, Ym = np.meshgrid(cm, cm[len(cm) // 2: len(cm) // 2 + 1])
norm = sim.locked.max_intensity(*np.meshgrid(cm, cm))
a_cam.plot(pxm, sim.locked.intensity(Xm, Ym)[0] / norm, color="k", lw=2.5, alpha=0.3, label="target")
l_free, = a_cam.plot([], [], color="#c0392b", lw=1, label="no lock")
l_lock, = a_cam.plot([], [], color="#0d8774", lw=1.6, ls="--", label="locked")
a_cam.set_xlim(pxm[0], pxm[-1]); a_cam.set_ylim(0, 1.15); a_cam.set_xlabel("x (pixel)"); a_cam.set_ylabel("I / I_max")
a_cam.set_title("PI-MAX cross-section (810 nm)"); a_cam.legend(fontsize=7, loc="upper right"); a_cam.grid(alpha=0.3)

a_txt.axis("off")
txt = a_txt.text(0.0, 1.0, "", va="top", ha="left", family="monospace", fontsize=9.5, transform=a_txt.transAxes)

hist = {k: [] for k in ("t", "pd", "ef", "el", "cmd", "pos")}


def render():
    buf = io.BytesIO(); fig.savefig(buf, format="png"); return IPImage(buf.getvalue())


handle = display(render(), display_id=True)
t_wall0 = time.time()
try:
    while sim.t < LIVE_DURATION_S - 1e-9:
        for _ in range(steps_per_frame):
            sim.step()
        snap = sim.snapshot()
        hist["t"].append(snap["t"]); hist["pd"].append(snap["pd"])
        hist["ef"].append(snap["err_free"]); hist["el"].append(snap["err_lock"])
        hist["cmd"].append(1e3 * (np.array(snap["cmd"]) - sim.theta0))
        hist["pos"].append(1e3 * (np.array(snap["theta"]) - sim.theta0))
        t = np.array(hist["t"]); lo = max(0.0, snap["t"] - WINDOW_S)
        pd, ef, el = np.array(hist["pd"]), np.unwrap(np.array(hist["ef"]), axis=0), np.array(hist["el"])
        cmd, pos = np.array(hist["cmd"]), np.array(hist["pos"])

        I_now = pinhole_intensity(sim.ref, xl, 0.0) / Inorm
        l_fr.set_ydata(I_now)
        for dot, x, v in zip(pd_dots, sim.x_ph, pinhole_intensity(sim.ref, sim.x_ph, 0.0) / Inorm):
            dot.set_data([1e3 * x], [v])
        for j in range(2):
            l_pd[j].set_data(t, pd[:, j]); l_ef[j].set_data(t, ef[:, j]); l_el[j].set_data(t, el[:, j])
            l_cmd[j].set_data(t, cmd[:, j]); l_pos[j].set_data(t, pos[:, j])
        m = t >= lo                                   # scroll the time plots, rescale around their centre line
        for a_, data, c0, min_half in ((a_pd, pd, 1.0, 0.02), (a_er, np.hstack([ef, el]), 0.0, 0.05),
                                       (a_th, np.hstack([cmd, pos]), 0.0, 0.05)):
            half = 1.15 * max(np.abs(data[m] - c0).max(), min_half)
            a_.set_xlim(lo, max(WINDOW_S, snap["t"])); a_.set_ylim(c0 - half, c0 + half)
        l_free.set_data(pxm, sim.free.intensity(Xm, Ym)[0] / norm)
        l_lock.set_data(pxm, sim.locked.intensity(Xm, Ym)[0] / norm)

        slow = (time.time() - t_wall0) / max(snap["t"], 1e-9)
        lines = [f"simulated time   {snap['t']:6.2f} s   ({slow:.1f}x slower than real)",
                 f"loop updates     {sim.n_updates:6d}        ({1 / (sim.every_loop * sim.dt):.0f} Hz)", "",
                 "photodiodes (/ setpoint):  " + "   ".join(f"PD{i+1} {v:6.3f}" for i, v in enumerate(snap["pd"])),
                 f"810 nm error:  φA−φC {1e3 * snap['err_lock'][0]:+7.1f} mrad   φB−φC {1e3 * snap['err_lock'][1]:+7.1f} mrad",
                 "", "XPS axes:"]
        for k, p in enumerate("AB"):
            lines.append(f"  slide {p}: {snap['theta'][k]:9.4f}°  {snap['status'][k]}")
        lines += ["", "events:"] + [f"  t = {te:5.2f} s  {msg}" for te, msg in sim.events[-4:]]
        txt.set_text("\n".join(lines))
        handle.update(render())
        if FRAME_DELAY_S:
            time.sleep(FRAME_DELAY_S)
except KeyboardInterrupt:
    print(f"stopped at t = {sim.t:.2f} s")
